![Week 4 Day 3](../../assets/banners/w4d3.png)

# Week 4 Day 3 - Solution: Pricing a Whole Class

The exercise: for every synthetic sheet, render each page at 150, 200 and 300 DPI, pre-process it, add up the image-token estimates under both rules, and price the class. Then repeat with a 2400 px size cap.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Try the exercise yourself before reading on: struggling a little is where the learning happens! These are <i>reference</i> solutions, not the only right answers.</span>
        </td>
    </tr>
</table>

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("❌ OpenAI API Key not set. Most of today's cells still run offline; the few that call a model will")
    print("   show a friendly 'Missing API key' panel. See Part 5 of the setup guide, or setup/SWAP_PROVIDER.md")

In [ ]:
import json

from gradewise.config import DATA_DIR, OUTPUT_DIR, REPO_ROOT
from gradewise.console import success, warn

SYNTHETIC = DATA_DIR / "answer-sheets" / "synthetic"
SHEETS = {p.stem: p for p in sorted(SYNTHETIC.glob("student_*.pdf"))}
TRUTH_FILE = SYNTHETIC / "transcripts.json"
TRUTH = json.loads(TRUTH_FILE.read_text(encoding="utf-8")) if TRUTH_FILE.exists() else {}

if SHEETS and TRUTH:
    success("Synthetic answer sheets found", {"Folder": str(SYNTHETIC.relative_to(REPO_ROOT)), "Sheets": len(SHEETS)})
else:
    # ⚠️ Plan B: the sheets aren't there yet (run `uv run scripts/make_synthetic_sheets.py`, or `git pull`).
    #    We draw a small stand-in sheet so the rest of the lab still runs. Its text is our "ground truth".
    from PIL import Image, ImageDraw, ImageFont

    warn("data/answer-sheets/synthetic/ is missing or incomplete: using a small stand-in sheet instead.")
    standin = {
        "Q3": "Dedlock is when a process waits for a long time. Conditions: mutual exclution and circuler wait.",
        "Q4a": "When two threads use the same varible at the same time.",
    }
    try:
        font = ImageFont.truetype("C:/Windows/Fonts/Inkfree.ttf", 34)
    except OSError:
        font = ImageFont.load_default()
    pages = []
    for label, qid in [("3)", "Q3"), ("4a)", "Q4a")]:
        img = Image.new("RGB", (1240, 1754), "white")
        draw = ImageDraw.Draw(img)
        words, lines, line = f"{label} {standin[qid]}".split(), [], ""
        for word in words:
            if len(line) + len(word) > 48:
                lines.append(line)
                line = ""
            line += word + " "
        for i, text in enumerate([*lines, line]):
            draw.text((90, 140 + i * 70), text, fill=(28, 45, 140), font=font)
        pages.append(img)
    target = OUTPUT_DIR / "week4" / "student_standin.pdf"
    target.parent.mkdir(parents=True, exist_ok=True)
    pages[0].save(target, "PDF", resolution=150, save_all=True, append_images=pages[1:])
    SHEETS, TRUTH = {"student_standin": target}, {"student_standin": standin}

In [ ]:
def show_table(rows: list[dict]) -> None:
    """Render a list of dicts as a Markdown table (a tiny stand-in for pandas: no extra dependencies)."""
    if not rows:
        print("(no rows)")
        return
    cols = list(rows[0])

    def cell(value) -> str:
        return str(value).replace("|", "\\|").replace("\n", " ⏎ ")

    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(cell(r.get(c, "")) for c in cols) + " |" for r in rows]
    display(Markdown("\n".join(lines)))

In [ ]:
import time

from gradewise.transcription import preprocess, render_pages
from gradewise.transcription.graph import ROUGH_PRICE_PER_M_INPUT
from gradewise.transcription.pdf_to_images import estimate_image_tokens

PROMPT_TOKENS = 900  # the system prompt + page prompt, roughly


def area_tokens(img, long_edge: int = 1568) -> int:
    scale = min(1.0, long_edge / max(img.size))
    return round(img.width * scale * img.height * scale / 750)


rows = []
for max_side in (1600, 2400):
    for dpi in (150, 200, 300):
        start = time.perf_counter()
        pages = [preprocess(img, max_side=max_side) for path in SHEETS.values() for img in render_pages(path, dpi=dpi)]
        seconds = time.perf_counter() - start
        tiles = sum(estimate_image_tokens(p) for p in pages) + PROMPT_TOKENS * len(pages)
        area = sum(area_tokens(p) for p in pages) + PROMPT_TOKENS * len(pages)
        rows.append(
            {
                "cap": max_side,
                "DPI": dpi,
                "pages": len(pages),
                "tiles tokens": f"{tiles:,}",
                "tiles $": f"{tiles / 1e6 * ROUGH_PRICE_PER_M_INPUT:.3f}",
                "area tokens": f"{area:,}",
                "area $": f"{area / 1e6 * ROUGH_PRICE_PER_M_INPUT:.3f}",
                "render + preprocess": f"{seconds:.1f} s",
            }
        )
show_table(rows)

### What the numbers say

- **DPI barely moves the bill.** From 150 to 300 DPI the token estimates are almost flat, because every page is capped (by us, then by the provider) before it's counted. What *does* grow with DPI is the **render time**, on your machine, for every page of every script.
- **The size cap is the real lever.** Under the area rule, raising the cap from 1600 to 2400 px would cost more, except the provider downsizes anything over ~1568 px anyway, so you pay in upload time for pixels the model never sees. Under the tiles rule it makes no difference at all.
- **Images aren't the whole bill.** The ~900 prompt tokens per page are a big share of the input. 🚀 **PRO:** prompt caching (supported by several providers for a repeated system prompt) can cut that part substantially.
- And remember output tokens, which this estimate leaves out: they are usually priced several times higher than input.

So for Asha's 10 scripts the whole first pass costs cents; for 200 scripts, still well under a few dollars. The expensive mistakes are elsewhere: a transcriber that "fixes" answers costs far more in unfair marks than any DPI setting.